In [ ]:
import sys
sys.path.insert(0, '../')

In [ ]:
import os
import cv2
import numpy as np
import pickle
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from pose_estimation import PoseEstimator

DATA_DIR = 'data'
MODEL_PATH = '../models/shot_classifier.pkl'
CLASSES = ['backhand', 'forehand', 'serve']

## Extract keypoints from labeled images

In [ ]:
estimator = PoseEstimator()

X, y = [], []
skipped = 0

for label, class_name in enumerate(CLASSES):
    folder = os.path.join(DATA_DIR, class_name)
    for fname in os.listdir(folder):
        if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        frame = cv2.imread(os.path.join(folder, fname))
        keypoints = estimator.get_keypoints(frame)
        if keypoints is None:
            skipped += 1
            continue
        X.append(keypoints)
        y.append(label)

X = np.array(X)
y = np.array(y)
print(f'Samples: {len(X)}  |  Skipped (no pose): {skipped}')
print(f'Class counts: { {CLASSES[i]: int((y == i).sum()) for i in range(len(CLASSES))} }')

## Train XGBoost

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, stratify = y, random_state = 42)

model = XGBClassifier(n_estimators = 100, max_depth = 4, learning_rate = 0.1, eval_metric = 'mlogloss')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names = CLASSES))

## Save model

In [ ]:
with open(MODEL_PATH, 'wb') as f:
    pickle.dump(model, f)
print(f'Saved to {MODEL_PATH}')